# Multimodal RAG — Text + Images with Vision-Language Models
## A Hands-On Workshop

**Multimodal RAG** extends classical text-only retrieval-augmented generation to handle **images alongside documents**. Instead of embedding only text, you encode images into the retrieval index — either via vision-language captions or CLIP embeddings — then answer natural-language questions that require understanding both modalities.

By the end of this notebook you will understand *why* multimodal retrieval is harder than text-only RAG, *which* image-encoding strategy to choose for a given task, and *how* to wire a production-grade pipeline with GPT-4o vision, ChromaDB, and LangGraph.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Multimodal RAG vs text-only RAG |
| 2 | **Image Encoding Strategies** — CLIP, caption-then-embed, GPT-4V |
| 3 | **Vision API** — Base64 images with `image_url` content blocks |
| 4 | **Indexing Pipeline** — Describe images, embed, store in ChromaDB |
| 5 | **Query Pipeline** — Similarity search + VLM answer synthesis |
| 6 | **LangGraph Workflow** — Connecting nodes: describe → store → answer |
| 7 | **Debugging & Evaluation** — Inspect retrievals, score answer quality |
| ★ | **Advanced Techniques** — Hybrid search, metadata, MMR |

---

### Prerequisites
- Python 3.10+ with a `.venv` that has `requirements.txt` installed, **or** run on Google Colab (the install cell below handles everything)
- An `OPENAI_API_KEY` with GPT-4o / GPT-4o-mini access (vision is enabled by default on both)

### Key References
> Lewis, P., Perez, E., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401
>
> Radford, A., Kim, J. W., et al. (2021). *Learning Transferable Visual Models From Natural Language Supervision (CLIP).* ICML 2021. https://arxiv.org/abs/2103.00020
>
> Liu, H., Li, C., Wu, Q., and Lee, Y. J. (2023). *Visual Instruction Tuning (LLaVA).* NeurIPS 2023. https://arxiv.org/abs/2304.08485
>
> Liu, N. F., et al. (2023). *Lost in the Middle: How Language Models Use Long Contexts.* https://arxiv.org/abs/2307.03172

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "langchain",
            "langchain-openai",
            "langchain-community",
            "langchain-chroma",
            "chromadb",
            "langgraph",
            "httpx",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import base64
import math
import os
from typing import List, TypedDict

import httpx

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

# ── Core imports ──────────────────────────────────────────────────────────────
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.graph import END, StateGraph

# ── Sanity check ──────────────────────────────────────────────────────────────
api_key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(api_key) and api_key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid format: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running any vision or embedding cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — What Is Multimodal RAG and Why Does It Exist?

---

### The Gap in Text-Only RAG

Classical RAG indexes **text**. But real document collections contain:
- Product photos, architectural diagrams, charts, infographics
- Scanned documents with embedded images
- Slide decks where the key information lives in a figure, not a caption
- Medical images, satellite imagery, technical drawings

A text-only retriever simply **cannot see** any of this content. Multimodal RAG bridges that gap by bringing images into the retrieval index.

---

### Three Approaches Compared

| Approach | Cost | Query quality | Indexes image text? | Setup complexity |
|----------|------|--------------|---------------------|------------------|
| **Text-only RAG** (ignore images) | Low | Poor on image content | No | Trivial |
| **CLIP embeddings** (image vectors) | Low | Medium (visual similarity) | Partially | Medium |
| **Caption-then-embed** (this notebook) | Medium (1 VLM call/image) | High (language-space) | Yes | Medium |
| **GPT-4V at query time** (no index) | High (VLM per query) | Highest | Yes | Low |

Caption-then-embed wins for most production multimodal RAG scenarios: it pays the VLM cost once at index time and then queries cheaply with text embeddings.

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **VLM / Vision-Language Model** | LLM that accepts both image and text as input (e.g. GPT-4o, GPT-4o-mini, LLaVA) |
| **CLIP** | Contrastive Language-Image Pretraining — maps images and text into a shared embedding space |
| **image_url content block** | OpenAI API format for passing an image to a VLM (base64 data URI or HTTPS URL) |
| **Caption** | A rich text description of an image generated by a VLM at index time |
| **Modality fusion** | Combining retrieval signals from multiple data types (text, image, audio) in one pass |
| **MultimodalState** | TypedDict flowing through LangGraph nodes carrying image metadata, captions, and answers |

### Multimodal RAG Architecture

```
OFFLINE — INDEX PHASE (run once when image collection changes)
─────────────────────────────────────────────────────────────────────────

  Text documents          Image files / URLs
       │                        │
       ▼                        ▼
  ┌──────────────┐    ┌────────────────────┐
  │ Text Splitter│    │   Vision LLM       │  GPT-4o-mini generates
  └──────┬───────┘    │  (describe images) │  a 2-3 sentence caption
         │            └────────┬───────────┘  per image
         │                     │
         ▼                     ▼
  ┌──────────────────────────────────────┐
  │       Text Embedding Model           │  Both text chunks AND
  │    (text-embedding-3-small)          │  image captions become
  └──────────────────┬───────────────────┘  1536-dim numeric vectors
                     │
                     ▼
  ┌──────────────────────────────────────┐
  │       ChromaDB Vector Store          │  Unified index of text
  │   (in-memory or persistent on disk)  │  chunks + image captions
  └──────────────────────────────────────┘


ONLINE — QUERY PHASE (every user question)
─────────────────────────────────────────────────────────────────────────

  User: "What does the dashboard diagram show?"
       │
       ▼
  ┌──────────────────┐
  │  Embedding Model │  query → query vector
  └────────┬─────────┘
           │
           ▼
  ┌──────────────────┐
  │   ChromaDB       │  cosine similarity search → top-k captions
  │  Vector Store    │  (retrieved docs may be text OR image captions)
  └────────┬─────────┘
           │
           ▼
  ┌──────────────────────────────────────────┐
  │  Prompt = system + retrieved captions    │  Multimodal context
  │          + original user query           │  in a single text prompt
  └────────┬─────────────────────────────────┘
           │
           ▼
  ┌──────────────────┐
  │   Answer LLM     │  generates grounded answer from retrieved captions
  └────────┬─────────┘
           │
           ▼
  "The dashboard diagram shows three panels: ..."
```

> Architecture pattern based on Lewis et al. (2020) RAG + Liu et al. (2023) LLaVA visual instruction tuning.

In [ ]:
# ─── 1-A: Why text-only RAG fails on image content ────────────────────────────
# Demonstrate the core problem: a text retriever returns nothing useful
# when the knowledge lives in an image.

# Pretend we have text descriptions of images — the WRONG way (manual, incomplete)
bad_index = [
    "Figure 1 — see attached image.",
    "Image 2 — diagram of system architecture.",
    "Photo 3 — product photo.",
]

query = "What animals are in the collection?"

print("Text-only index with vague captions (no vision LLM used):")
for i, entry in enumerate(bad_index):
    print(f"  [{i}] {entry}")
print()
print(f"Query: '{query}'")
print("Result: Cannot retrieve anything useful — no animal content in index.")
print()
print("Solution: Use a vision LLM to generate rich captions at index time.")
print("Then the same image entry becomes:")
print()
print("  [cat] A domestic tabby cat with brown and black striped fur sits")
print("        upright on a rustic wooden surface, gazing alertly to the left.")
print("        The cat has bright yellow-green eyes and fluffy, dense fur.")
print()
print("Now queries about 'animals', 'cats', 'stripes', 'eyes' all match correctly.")

---

## Part 2 — Image Encoding Strategies

---

### The Three Strategies in Detail

```
Strategy A — CLIP (direct image embeddings)
────────────────────────────────────────────
  image → CLIP vision encoder → 512-d vector → vector store

  PRO:  No LLM API call at index time — free and fast.
  PRO:  Works for purely visual similarity (find images similar to this one).
  CON:  Embedding is in image-space; natural-language text queries must also
        be CLIP-encoded for the comparison to be valid.
  CON:  Misses fine-grained detail: text inside images, exact counts,
        spatial relationships ("the dog is to the LEFT of the cat").


Strategy B — Caption-then-embed  [used in this notebook]
────────────────────────────────────────────────────────
  image → vision LLM → text caption → text embedding → vector store

  PRO:  Captions are in language-space — same space as user queries.
        Standard text embeddings (text-embedding-3-small) work perfectly.
  PRO:  Rich detail: colors, objects, OCR text, spatial layout, mood.
  PRO:  Compatible with any text vector store — no special CLIP index.
  CON:  One VLM API call per image at index time (cost + latency).
  CON:  Caption quality depends on the VLM — GPT-4o > GPT-4o-mini
        for complex technical diagrams.


Strategy C — GPT-4V at query time (no index)
────────────────────────────────────────────
  query + ALL images → VLM per query → answer

  PRO:  Zero indexing cost — describe and answer on demand.
  PRO:  Highest answer quality — model sees the original pixels.
  CON:  Does not scale — O(n) VLM calls per query.
  CON:  Context window limits total image collection size.
  USE:  Only practical for very small collections (< 10 images).
```

### CLIP vs Caption-then-embed: When to Choose Which

| Dimension | CLIP | Caption-then-embed |
|-----------|------|--------------------|
| Index cost | $0 (local model) | ~$0.001/image (GPT-4o-mini) |
| Query cost | Same as text embedding | Same as text embedding |
| Query language | Works best with short noun phrases | Works with full natural-language questions |
| OCR / text in images | Poor | Excellent |
| Diagram / chart understanding | Partial | Good to Excellent (GPT-4o) |
| Collection update speed | Fast | Slower (LLM per image) |
| Best for | Visual similarity search, e-commerce | Question-answering over image collections |

In [ ]:
# ─── 2-A: Caption quality comparison ─────────────────────────────────────────
# Show that richer captions dramatically expand what queries can retrieve.
# We simulate two caption styles without making actual API calls.

CAPTION_MINIMAL = (
    "[cat] A cat is shown in the image."
)

CAPTION_RICH = (
    "[cat] A domestic tabby cat with brown and black striped fur sits upright "
    "on a rustic wooden surface, gazing alert and attentive to the left. "
    "The background shows a blurred outdoor scene with soft green tones. "
    "The cat appears calm and healthy, with bright yellow-green eyes."
)

queries = [
    "Is there a cat?",
    "What color are the cat's eyes?",
    "Is the animal indoors or outdoors?",
    "What is the cat sitting on?",
    "Is the cat aggressive or calm?",
]

print("Caption quality impact on retrievability\n")
print(f"MINIMAL: '{CAPTION_MINIMAL}'")
print(f"RICH:    '{CAPTION_RICH[:80]}...'")
print()
print(f"{'Query':<45} {'Minimal':>8} {'Rich':>6}")
print("-" * 62)

minimal_answers = [True, False, False, False, False]
rich_answers    = [True, True, True, True, True]

for q, m, r in zip(queries, minimal_answers, rich_answers):
    m_str = "YES" if m else "NO "
    r_str = "YES" if r else "NO "
    print(f"{q:<45} {m_str:>8} {r_str:>6}")

print()
print("Lesson: Rich, detailed captions dramatically expand retrieval coverage.")
print("Use a capable model (GPT-4o) for complex images; GPT-4o-mini for simple ones.")

---

## Part 3 — The Vision API: `image_url` Content Blocks

---

### How OpenAI's Vision API Works

GPT-4o and GPT-4o-mini accept **multi-part messages** where each part is either `text` or `image_url`. An `image_url` part can contain:
- A public HTTPS URL pointing to a JPEG, PNG, GIF, or WebP
- A **base64-encoded** data URI: `data:image/jpeg;base64,<data>`

Base64 encoding is preferred when: the image is private, the URL might expire, or you need deterministic behavior (URLs can return different content over time).

```python
# Structure of a vision message (LangChain)
HumanMessage(
    content=[
        {
            "type": "text",
            "text": "Describe this image in detail."
        },
        {
            "type": "image_url",
            "image_url": {
                "url": "data:image/jpeg;base64,<base64_string>",
                "detail": "high"   # "low" (~85 tokens) or "high" (up to 1700 tokens)
            }
        }
    ]
)
```

### Image Token Cost

| Detail level | Approx. token cost | Use for |
|--------------|--------------------|---------|
| `low` | ~85 tokens | Thumbnails, simple scenes, quick classification |
| `high` | 170–1700 tokens | Complex diagrams, OCR, technical drawings |
| `auto` (default) | Model chooses | Most general-purpose indexing |

In [ ]:
# ─── 3-A: Fetch an image and convert to base64 ────────────────────────────────

def fetch_image_b64(url: str) -> str:
    """Download an image from a URL and return it as a base64 string."""
    resp = httpx.get(url, timeout=15, follow_redirects=True)
    resp.raise_for_status()
    return base64.b64encode(resp.content).decode()


CAT_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "4/4d/Cat_November_2010-1a.jpg/320px-Cat_November_2010-1a.jpg"
)

try:
    cat_b64 = fetch_image_b64(CAT_URL)
    print(f"Image fetched successfully.")
    print(f"Base64 length: {len(cat_b64):,} characters")
    print(f"Approx. size:  {len(cat_b64) * 3 / 4 / 1024:.1f} KB (original bytes)")
    print(f"First 60 chars of base64: {cat_b64[:60]}...")
    print()
    print("This base64 string is embedded in the image_url content block as:")
    print('  {"url": "data:image/jpeg;base64,<base64_string>"}')
except Exception as e:
    cat_b64 = None
    print(f"Image fetch failed (offline?): {e}")
    print("Continuing — subsequent cells handle the None case gracefully.")

In [ ]:
# ─── 3-B: Call the vision LLM with an image ───────────────────────────────────
# This is the core indexing operation: one image → one rich text caption.

vision_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, max_tokens=300)


def describe_image(b64: str, label: str = "", detail: str = "auto") -> str:
    """Send a base64-encoded image to GPT-4o-mini and return a rich caption."""
    prompt_text = (
        "Describe this image in 3-4 sentences for use in a search index. "
        "Include: main subjects, colors, setting, any visible text, and mood. "
        f"Label hint: {label}." if label else
        "Describe this image in 3-4 sentences for use in a search index. "
        "Include: main subjects, colors, setting, any visible text, and mood."
    )
    msg = HumanMessage(
        content=[
            {"type": "text", "text": prompt_text},
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{b64}",
                    "detail": detail,
                },
            },
        ]
    )
    result = vision_llm.invoke([msg])
    return result.content


if cat_b64:
    cat_caption = describe_image(cat_b64, label="a domestic cat")
    print("Caption generated by GPT-4o-mini:")
    print()
    print(cat_caption)
else:
    cat_caption = "A tabby cat with striped brown and black fur sits on a wooden surface."
    print("Using offline fallback caption (no internet connection).")

In [ ]:
# ─── 3-C: Compare `low` vs `high` detail ─────────────────────────────────────
# low  = ~85 tokens per image  — faster, cheaper, suitable for simple photos
# high = up to 1700 tokens     — for diagrams, OCR, detailed technical images

if cat_b64:
    print("=== low detail (~85 tokens per image) ===")
    low_caption = describe_image(cat_b64, label="a domestic cat", detail="low")
    print(low_caption)
    print()
    print("=== high detail (up to 1700 tokens per image) ===")
    high_caption = describe_image(cat_b64, label="a domestic cat", detail="high")
    print(high_caption)
    print()
    print(f"Word count — low: {len(low_caption.split())}, high: {len(high_caption.split())}")
    print()
    print("Rule of thumb:")
    print("  'low'  → product thumbnails, simple portraits, < 10 distinct visual elements.")
    print("  'high' → technical diagrams, screenshots with text, multi-panel figures.")
else:
    print("Skipped — image not available offline.")

### Exercise 1 — Describe Your Own Image

Replace `MY_IMAGE_URL` below with any publicly accessible image URL (JPEG or PNG). Experiment with the prompt text and the `detail` parameter.

**Goal:** Write a captioning prompt that produces output you could use to find this image with a natural-language question like *"show me images with red objects"* or *"which images contain text?"*

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
MY_IMAGE_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "a/a7/Camponotus_flavomarginatus_ant.jpg/"
    "640px-Camponotus_flavomarginatus_ant.jpg"
)

# TODO: replace with your own URL

# TODO: write a prompt that captures the specific details a user might search for
MY_PROMPT = (
    "Describe this image in 3-4 sentences for use in a search index. "
    "Cover: main subject and its color, the setting, any text visible, "
    "and at least one unique detail that distinguishes this from similar images."
)

# TODO: try 'low', 'high', and 'auto' — compare the outputs
MY_DETAIL = "auto"

try:
    my_b64 = fetch_image_b64(MY_IMAGE_URL)
    my_msg = HumanMessage(
        content=[
            {"type": "text", "text": MY_PROMPT},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{my_b64}", "detail": MY_DETAIL}},
        ]
    )
    my_caption = vision_llm.invoke([my_msg])
    print("Your caption:")
    print(my_caption.content)
except Exception as e:
    print(f"Could not fetch image: {e}")

---

## Part 4 — The Indexing Pipeline: Describe, Embed, Store

---

### Pipeline Components

```
IMAGE_CATALOGUE
  [{id, url, label}, ...]            ← your image metadata
       │
       ▼
  [fetch_image_b64]                  ← download + base64-encode each image
       │
       ▼
  [vision_llm.invoke per image]      ← GPT-4o-mini generates a rich caption
       │   (one API call per image — the VLM cost is paid once here)
       ▼
  captions: list[str]                ← e.g. "[cat] A tabby cat with brown..."
       │
       ▼
  [embeddings_model.embed_documents] ← text-embedding-3-small → 1536-d vectors
       │
       ▼
  [Chroma.from_texts]                ← vectors + captions + metadata in ChromaDB
       │
       ▼
  vectorstore (ready for similarity_search)
```

The **image never enters the vector store** — only its text caption does. This means any standard text embedding model and any text vector store works out of the box.

In [ ]:
# ─── 4-A: Define the image catalogue ──────────────────────────────────────────
# In production: load from a database, S3 listing, or a manifest YAML.

IMAGE_DOCS = [
    {
        "id": "cat",
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/320px-Cat_November_2010-1a.jpg",
        "label": "a domestic cat",
    },
    {
        "id": "dog",
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg",
        "label": "a yellow Labrador Retriever",
    },
    {
        "id": "coffee",
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/45/A_small_cup_of_coffee.JPG/320px-A_small_cup_of_coffee.JPG",
        "label": "a cup of espresso coffee",
    },
    {
        "id": "mountain",
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e7/Everest_North_Face_toward_Base_Camp_Tibet_Luca_Galuzzi_2006.jpg/320px-Everest_North_Face_toward_Base_Camp_Tibet_Luca_Galuzzi_2006.jpg",
        "label": "a snow-capped mountain landscape",
    },
    {
        "id": "books",
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a4/Bookshelf_in_library.jpg/320px-Bookshelf_in_library.jpg",
        "label": "a bookshelf in a library",
    },
]

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

print(f"Image catalogue: {len(IMAGE_DOCS)} images")
for doc in IMAGE_DOCS:
    print(f"  [{doc['id']:10}] {doc['label']}")

In [ ]:
# ─── 4-B: Generate captions for all images ────────────────────────────────────
# Index-time VLM calls. In production: run offline, cache results, re-run only
# when new images are added. Do NOT re-caption on every query.

CAPTION_PROMPT = (
    "Describe this image in 2-3 sentences for use in a search index. "
    "Include: main subjects, colors, setting, any visible text, and notable details. "
    "Be specific — vague captions reduce retrieval quality. "
    "Label hint: {label}"
)


def caption_image(doc: dict) -> str:
    """Fetch image, call vision LLM, return caption. Graceful fallback on error."""
    try:
        b64 = fetch_image_b64(doc["url"])
        msg = HumanMessage(
            content=[
                {"type": "text", "text": CAPTION_PROMPT.format(label=doc["label"])},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
            ]
        )
        result = vision_llm.invoke([msg])
        return f"[{doc['id']}] {result.content}"
    except Exception as e:
        # Graceful degradation: label-only caption keeps the image findable by label
        return f"[{doc['id']}] {doc['label']} (caption unavailable: {e})"


print("Generating captions (one VLM call per image)...\n")
captions = []
for doc in IMAGE_DOCS:
    caption = caption_image(doc)
    captions.append(caption)
    snippet = caption[len(doc['id']) + 3:len(doc['id']) + 153]
    print(f"  [{doc['id']}] {snippet}")
    print()

print(f"Done. {len(captions)} captions ready for embedding.")

In [ ]:
# ─── 4-C: Embed captions and build the vector store ───────────────────────────

# Metadata: one dict per image so every retrieved doc links back to its source
metadatas = [
    {"image_id": doc["id"], "url": doc["url"], "label": doc["label"]}
    for doc in IMAGE_DOCS
]

vectorstore = Chroma.from_texts(
    texts=captions,
    embedding=embeddings_model,
    metadatas=metadatas,
    collection_name="multimodal_image_index",
)

print("Vector store ready.")
print(f"  Collection: multimodal_image_index")
print(f"  Documents:  {vectorstore._collection.count()}")
print(f"  Embedding:  text-embedding-3-small (1536 dims)")
print()
print("Index is in-memory (Chroma EphemeralClient).")
print("For persistence across sessions, add persist_directory='./chroma_images'")
print("to the Chroma.from_texts() call above.")

---

## Part 5 — The Query Pipeline: Retrieve and Answer

---

### How LCEL Wires the Query Pipeline

```python
multimodal_rag_chain = (
    {"context": retriever | format_captions, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
```

| Component | Role | Input → Output |
|-----------|------|----------------|
| `retriever` | Cosine similarity search over caption embeddings | string → list of Documents |
| `format_captions` | Joins captions + image IDs into one context string | list of Documents → string |
| `RunnablePassthrough()` | Passes the question through unchanged | string → string |
| `prompt` | Fills `{context}` and `{question}` in the template | dict → ChatPromptValue |
| `llm` | Generates the grounded answer | ChatPromptValue → AIMessage |
| `StrOutputParser()` | Extracts plain text from AIMessage | AIMessage → string |

In [ ]:
# ─── 5-A: Build the retriever and inspect what it returns ─────────────────────
# ALWAYS inspect retrieved captions before trusting the LLM's answer.
# If retrieval is wrong, the answer will be wrong regardless of prompt quality.

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)


def show_retrieval(query: str):
    """Print what the retriever returns before the LLM ever sees it."""
    docs = retriever.invoke(query)
    print(f"Query: '{query}'")
    print(f"Retrieved {len(docs)} captions:\n")
    for i, doc in enumerate(docs, 1):
        meta = doc.metadata
        print(f"  [{i}] image_id={meta.get('image_id')}  label='{meta.get('label')}'")
        print(f"       {doc.page_content[:200]}")
    print()


show_retrieval("What animals are in the collection?")
show_retrieval("Is there a hot drink?")
show_retrieval("Show me outdoor landscapes.")

In [ ]:
# ─── 5-B: Build the full RAG chain and run queries ────────────────────────────

qa_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

system_prompt = (
    "You are an assistant that answers questions about an image collection. "
    "The context below contains text descriptions (captions) of images. "
    "Answer the user's question using ONLY the information in these captions. "
    "If the captions do not contain enough information, say so clearly. "
    "When relevant, cite which image (by id in brackets) your answer is based on.\n\n"
    "Captions:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])


def format_captions(docs: list) -> str:
    """Format retrieved caption Documents into a single context string."""
    lines = []
    for doc in docs:
        img_id = doc.metadata.get("image_id", "?")
        lines.append(f"Image [{img_id}]: {doc.page_content}")
    return "\n\n".join(lines)


rag_chain = (
    {"context": retriever | format_captions, "question": RunnablePassthrough()}
    | prompt
    | qa_llm
    | StrOutputParser()
)

test_queries = [
    "What animal is in the collection?",
    "Is there a beverage shown? What kind?",
    "Which images show outdoor scenes?",
    "What color is the dog?",
    "Are there any images of food or drink?",
    "Is there a helicopter in any image?",   # out of scope
]

print("=" * 60)
for q in test_queries:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print()

### Exercise 2 — Add More Images and Query the Expanded Index

Extend `IMAGE_DOCS` with 2-3 new entries (any public image URLs), re-run cells 4-B and 4-C, then run at least 3 new queries.

**Questions to investigate:**
1. Did the new images get retrieved for relevant queries?
2. Did adding more images cause any irrelevant captions to be retrieved?
3. What happens if you add two very similar images — two different cats, for example?

**Starter template below:**

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────
# Step 1: add entries here
EXTRA_IMAGES = [
    {
        "id": "TODO_id_1",
        "url": "https://TODO_replace_with_any_public_jpeg_url",
        "label": "TODO: one sentence description of what this image shows",
    },
    {
        "id": "TODO_id_2",
        "url": "https://TODO_replace_with_any_public_jpeg_url",
        "label": "TODO: one sentence description",
    },
]

# Step 2: extend the catalogue and re-run cells 4-B and 4-C
# IMAGE_DOCS.extend(EXTRA_IMAGES)

# Step 3: run new queries that your extra images can answer
NEW_QUERIES = [
    "TODO: a question your new images can answer",
    "TODO: a question that spans old + new images",
    "TODO: a question no image can answer (should get a graceful decline)",
]

print("Fill in EXTRA_IMAGES, uncomment IMAGE_DOCS.extend, then re-run from cell 4-B.")

---

## Part 6 — LangGraph Workflow: Describe → Store → Answer

---

### Why LangGraph?

The three-step pipeline maps naturally onto a **directed graph** where each node is a pure function that reads from and writes to a shared `State` TypedDict. LangGraph:
- Makes the data flow explicit and inspectable — you can see exactly what flows between nodes
- Makes it easy to insert new nodes (e.g., a grading node between `store` and `answer`)
- Supports streaming, persistence, and human-in-the-loop checkpoints for production use

### Graph Structure

```
START
  │
  ▼
[describe]  ──  fetches each image, calls vision LLM, writes to state.descriptions
  │
  ▼
[store]     ──  embeds state.descriptions, creates ChromaDB vectorstore
  │
  ▼
[answer]    ──  similarity_search(state.query, k=3) → QA LLM → state.answer
  │
  ▼
 END
```

### State Schema

```python
class MultimodalState(TypedDict):
    descriptions: list[str]   # written by [describe], read by [store]
    query: str                # provided by the caller, read by [answer]
    answer: str               # written by [answer], returned to caller
```

In [ ]:
# ─── 6-A: Define state, nodes, and compile the graph ─────────────────────────

class MultimodalState(TypedDict):
    descriptions: List[str]
    query: str
    answer: str


# Written by node_store, read by node_answer.
# In production: use a persistent directory and load it in node_answer instead.
_graph_vectorstore: Chroma | None = None


def node_describe(state: MultimodalState) -> dict:
    """Node 1: Fetch each image and generate a caption with the vision LLM."""
    descriptions = []
    for doc in IMAGE_DOCS:
        caption = caption_image(doc)
        descriptions.append(caption)
        print(f"  [describe] {doc['id']}: {caption[:90]}...")
    return {"descriptions": descriptions}


def node_store(state: MultimodalState) -> dict:
    """Node 2: Embed captions and load into ChromaDB."""
    global _graph_vectorstore
    _graph_vectorstore = Chroma.from_texts(
        texts=state["descriptions"],
        embedding=embeddings_model,
        metadatas=[
            {"image_id": doc["id"], "url": doc["url"]} for doc in IMAGE_DOCS
        ],
        collection_name="graph_image_index",
    )
    print(f"  [store] {_graph_vectorstore._collection.count()} captions indexed.")
    return {}


def node_answer(state: MultimodalState) -> dict:
    """Node 3: Retrieve top-k captions, call QA LLM, return answer."""
    if _graph_vectorstore is None:
        return {"answer": "Error: vector store not initialized."}
    docs = _graph_vectorstore.similarity_search(state["query"], k=3)
    context = "\n\n".join(
        f"Image [{d.metadata.get('image_id', '?')}]: {d.page_content}" for d in docs
    )
    qa_prompt = (
        "You answer questions about an image collection. "
        "Use ONLY the captions below. If the answer is not in the captions, say so.\n\n"
        f"{context}\n\nQuestion: {state['query']}\nAnswer:"
    )
    result = qa_llm.invoke([HumanMessage(content=qa_prompt)])
    print(f"  [answer] Retrieved {len(docs)} captions for: '{state['query']}'")
    return {"answer": result.content}


def create_workflow():
    graph = StateGraph(MultimodalState)
    graph.add_node("describe", node_describe)
    graph.add_node("store", node_store)
    graph.add_node("answer", node_answer)
    graph.set_entry_point("describe")
    graph.add_edge("describe", "store")
    graph.add_edge("store", "answer")
    graph.add_edge("answer", END)
    return graph.compile()


app = create_workflow()
print("Graph compiled: START → describe → store → answer → END")

In [ ]:
# ─── 6-B: Run the full LangGraph pipeline end-to-end ─────────────────────────
# Note: [describe] and [store] run on every invoke() — this is intentional for
# demonstration. In production, split into an index graph (runs once) and a
# query graph (runs per user question) to avoid re-captioning on every query.

GRAPH_QUERIES = [
    "What animals are in the collection?",
    "Is there a hot drink shown?",
    "Describe any outdoor scenes.",
]

for query in GRAPH_QUERIES:
    print(f"\n{'=' * 60}")
    print(f"QUERY: {query}")
    print("=" * 60)
    result = app.invoke({"descriptions": [], "query": query, "answer": ""})
    print(f"\nANSWER: {result['answer']}")

---

## Part 7 — Debugging and Evaluation

---

### Common Multimodal RAG Failure Modes

| Failure | Symptom | Root cause | Fix |
|---------|---------|-----------|-----|
| **Wrong image retrieved** | Answer references the wrong image | Vague caption | Use richer prompt; switch to `high` detail |
| **Model ignores captions** | Hallucinates image content | Weak system prompt | Add "use ONLY the captions" instruction |
| **Out-of-scope answered** | Model invents content not in index | No fallback instruction | Add "if not in captions, say so" |
| **Two similar images confused** | Wrong `[id]` cited | Captions too similar | Use more distinguishing prompts |
| **Vision API timeout** | `httpx.TimeoutException` | Large image + slow network | Increase timeout; resize images before encoding |
| **Base64 API error** | 400 Bad Request | Image > 20MB | Resize to max 1024px before encoding |

### Debugging Checklist
1. Print retrieved captions BEFORE calling the QA LLM — this is the most important step
2. Check similarity scores — low scores indicate poor caption quality, not just bad retrieval
3. Print the exact prompt sent to the QA LLM
4. Try k=1, k=3, k=5 — does answer quality change significantly?
5. Re-caption problematic images with a more specific prompt

In [ ]:
# ─── 7-A: Inspect retrieval scores ────────────────────────────────────────────
# similarity_search_with_score returns (Document, float) pairs.
# Chroma returns cosine DISTANCE — lower distance = more similar = better match.

def score_retrieval(query: str, k: int = 5):
    results = vectorstore.similarity_search_with_score(query, k=k)
    print(f"Query: '{query}'")
    print(f"{'Rank':<6} {'Similarity':>11} {'Image ID':<10} Caption (first 80 chars)")
    print("-" * 100)
    for rank, (doc, distance) in enumerate(results, 1):
        similarity = 1 - distance  # cosine distance → cosine similarity
        img_id = doc.metadata.get("image_id", "?")
        snippet = doc.page_content[:80].replace("\n", " ")
        bar = "█" * max(0, int(similarity * 20))
        print(f"  [{rank}]   {similarity:+.3f} {bar:<20}  [{img_id:<8}] {snippet}")
    print()


score_retrieval("What animals are in the collection?")
score_retrieval("Is there anything to drink?")
score_retrieval("Show me winter sports imagery")  # should score low — not in index

In [ ]:
# ─── 7-B: Experiment — how k affects answer quality ──────────────────────────
# More captions = more context. But past a collection's true diversity,
# extra retrieved captions add noise without helping.

test_query = "What animals are shown and what colors are they?"
print(f"Query: '{test_query}'\n")

for k_val in [1, 2, 3, 5]:
    docs = vectorstore.similarity_search(test_query, k=k_val)
    context = "\n".join(
        f"Image [{d.metadata.get('image_id', '?')}]: {d.page_content[:150]}"
        for d in docs
    )
    qa_msg = HumanMessage(
        content=(
            f"Answer using only the captions below. "
            f"If information is missing, say so.\n\n{context}\n\n"
            f"Question: {test_query}\nAnswer:"
        )
    )
    answer = qa_llm.invoke([qa_msg]).content
    print(f"k={k_val}: {answer[:250]}")
    print()

### Exercise 3 — Improve Caption Quality for a Hard Query

Pick a query that the current pipeline answers poorly (wrong image cited, answer too vague, or wrong details). Then:

1. Run `score_retrieval(HARD_QUERY)` and print the retrieved captions
2. Identify which image has a poor caption and why
3. Write a **more specific captioning prompt** for that image — add: exact colors, texture, counts, visible text, spatial layout
4. Re-caption that image, update the `captions` list, rebuild the vectorstore
5. Re-run the query and compare before/after answers

**Key insight:** The caption is the only bridge between visual content and natural-language queries. Every meaningful visual attribute should appear explicitly in the text if you want it to be queryable.

In [ ]:
# ── Exercise 3 Starter ────────────────────────────────────────────────────────

HARD_QUERY = "TODO: pick a query the current pipeline handles poorly"

# Step 1: inspect what gets retrieved
if not HARD_QUERY.startswith("TODO"):
    score_retrieval(HARD_QUERY, k=3)

# Step 2: write a better prompt targeting the specific weakness
BETTER_CAPTION_PROMPT = (
    "TODO: write a prompt that explicitly extracts the detail "
    "your hard query needs (color, spatial position, visible text, etc.)"
)

# Step 3: re-caption the specific image
# target_doc = IMAGE_DOCS[0]  # set to the image that needs better captioning
# better_caption = caption_image(target_doc)  # will use updated CAPTION_PROMPT above
# print(better_caption)

# Step 4: update captions list and rebuild vectorstore
# captions[0] = better_caption  # update at the correct index
# vectorstore = Chroma.from_texts(captions, embeddings_model, metadatas=metadatas)

# Step 5: re-run the query
# score_retrieval(HARD_QUERY, k=3)

print("Fill in HARD_QUERY and BETTER_CAPTION_PROMPT, then uncomment Steps 3-5.")

---

## Part 8 ★ — Advanced Techniques (Bonus)

---

### Separate Index and Query Phases

The graph in Part 6 re-captions every image on each `invoke()` — correct for this notebook but wasteful in production. The right architecture:

```python
# Run once — when images change
index_app = build_index_graph()    # describe + store nodes only
index_app.invoke({"descriptions": [], "query": "", "answer": ""})

# Run per query — load existing index
query_app = build_query_graph()    # load persistent Chroma + answer node
result = query_app.invoke({"query": "What animals are shown?"})
```

### Metadata Filtering — Search Within a Subset

```python
# Only retrieve captions tagged 'wildlife'
vectorstore.similarity_search(
    query,
    k=3,
    filter={"category": "wildlife"},
)
```

### Persistent Index

```python
# Persist to disk — survives Python restarts
vectorstore = Chroma.from_texts(
    texts=captions,
    embedding=embeddings_model,
    metadatas=metadatas,
    persist_directory="./chroma_image_index",
    collection_name="images_v1",
)

# Reload in a new session
vectorstore = Chroma(
    persist_directory="./chroma_image_index",
    embedding_function=embeddings_model,
    collection_name="images_v1",
)
```

In [ ]:
# ─── 8-A: MMR vs standard similarity on near-duplicate images ─────────────────
# When two captions are very similar, standard search returns both.
# MMR (Maximal Marginal Relevance) trades off relevance vs. diversity.

similar_captions = [
    "[cat1] A tabby cat with brown and black stripes sits on a wooden table. "
    "The cat has bright green eyes and fluffy, dense fur.",
    "[cat2] A brown and black striped tabby cat rests on a wooden surface. "
    "It has golden-green eyes and a calm, composed expression.",
    "[dog1] A golden Labrador Retriever sits on a green lawn, tail wagging. "
    "The dog has a warm, friendly expression and honey-colored fur.",
    "[coffee1] A small espresso in a white ceramic cup on a wooden saucer. Steam rises.",
    "[mountain1] Snow-capped peaks under a deep blue sky. A glacial valley below.",
]

mmr_store = Chroma.from_texts(similar_captions, embeddings_model)
mmr_query = "Tell me about the cats in the collection"

print("Standard similarity (k=3) — may return near-duplicate cat captions:")
for doc in mmr_store.similarity_search(mmr_query, k=3):
    print(f"  {doc.page_content[:100]}")

print()
print("MMR (k=3, lambda_mult=0.5) — balances relevance with diversity:")
for doc in mmr_store.max_marginal_relevance_search(
    mmr_query, k=3, fetch_k=5, lambda_mult=0.5
):
    print(f"  {doc.page_content[:100]}")

print()
print("lambda_mult: 0.0 = maximum diversity, 1.0 = maximum relevance (same as standard).")

In [ ]:
# ─── 8-B: Score threshold — decline gracefully for out-of-scope queries ───────
# Instead of always returning the k "best" captions even when nothing matches,
# filter by a minimum similarity score. Returns an empty list when no caption
# is above the threshold, which the QA LLM should interpret as "not in index".

threshold_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.6},
)

in_scope = [
    "What animals are in the collection?",
    "Is there a hot drink?",
]
out_of_scope = [
    "Show me images of cars and motorcycles",
    "Which images contain Python source code?",
]

for q in in_scope + out_of_scope:
    docs = threshold_retriever.invoke(q)
    tag = "IN-SCOPE " if docs else "OUT-SCOPE"
    print(f"[{tag}] '{q}'")
    if docs:
        print(f"  Retrieved: {[d.metadata.get('image_id') for d in docs]}")
    else:
        print("  Retrieved: [] — below threshold, QA LLM should decline gracefully")
    print()

---

## What's Next?

You now have a complete multimodal RAG pipeline. Here is where to go deeper:

### Improve retrieval quality
- **Richer captioning** — prompt the vision LLM to capture: object counts, exact colors, text in the image, spatial layout ("the red object is in the top-left corner"), and domain-specific details.
- **Hybrid search** — combine caption embeddings with BM25 keyword matching for collections where exact labels matter (e.g., product codes visible in images).
- **Re-ranking** — retrieve k=10 captions with embedding similarity, then use a cross-encoder to re-rank and keep the top 3.

### Scale the pipeline
- **Persistent index** — add `persist_directory` to `Chroma.from_texts` so captions survive restarts; use `.add_texts()` for incremental updates when new images arrive.
- **Async indexing** — caption images in parallel with `asyncio.gather` and `vision_llm.ainvoke` to index large collections quickly.
- **LangSmith tracing** — set `LANGCHAIN_TRACING_V2=true` to see every retrieval + generation call in a dashboard.

### Related examples in this repository
- **`examples/1-basic-local-rag`** — ChromaDB and text embedding fundamentals without the vision layer
- **`examples/12-basic-rag-notebook`** — Full text-only RAG workshop with debugging and evaluation
- **`examples/17-corrective-rag`** — Grade retrieved documents before generating an answer
- **`examples/25-adaptive-rag`** — Route queries to different retrieval strategies dynamically
- **`examples/30-agentic-rag`** — Agentic loop that decides when retrieval is needed

---

### Academic References

1. **Lewis, P., Perez, E., et al. (2020).** *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401

2. **Radford, A., Kim, J. W., et al. (2021).** *Learning Transferable Visual Models From Natural Language Supervision (CLIP).* ICML 2021. https://arxiv.org/abs/2103.00020

3. **Liu, H., Li, C., Wu, Q., and Lee, Y. J. (2023).** *Visual Instruction Tuning (LLaVA).* NeurIPS 2023. https://arxiv.org/abs/2304.08485

4. **Liu, N. F., et al. (2023).** *Lost in the Middle: How Language Models Use Long Contexts.* TACL 2024. https://arxiv.org/abs/2307.03172

5. **Gao, Y., Xiong, Y., et al. (2023).** *Retrieval-Augmented Generation for Large Language Models: A Survey.* https://arxiv.org/abs/2312.10997

---

*Workshop complete. Open `examples/12-basic-rag-notebook/rag_workbook.ipynb` for the text-only RAG deep dive, or `examples/17-corrective-rag` for graded retrieval.*

---

## Exercise Answer Key

Use this section after attempting the exercises yourself. These are sample solutions — not the only correct approaches.

### Exercise 1 — Describe Your Own Image

**What a good caption looks like:**
A caption that contains enough specific detail that a user asking a targeted question (color, subject type, setting, mood, visible text) would retrieve this image and not a different one.

**Sample prompt structure that works well:**
```
Describe this image in 3-4 sentences for use in a search index.
Cover: (1) main subject and its exact colors and texture,
       (2) the setting or background,
       (3) any visible text, labels, or brand names,
       (4) the mood, lighting, or context.
Be specific about what makes this image different from similar images.
```

**Common mistakes:**
- Single sentence prompt → too vague for precise retrieval
- No color instruction → color queries will fail
- No instruction to read text in the image → OCR-style queries miss
- Using `detail="low"` for diagrams → model sees a thumbnail, misses fine details

### Exercise 2 — Add More Images

**Expected behavior after extending the catalogue:**
- New images are retrieved when the query matches their captions
- Two very similar images (two cats) may both be retrieved for animal queries — use MMR (Part 8-A) to diversify
- Adding unrelated images should not degrade answers for unrelated queries, provided similarity scores remain below the relevant threshold

**Sample new images for easy testing:**
```python
{
    "id": "sunset",
    "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1e/Sunset_over_the_sea.jpg/320px-Sunset_over_the_sea.jpg",
    "label": "a sunset over the ocean",
},
{
    "id": "city",
    "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/0/05/Southwest_corner_of_Central_Park%2C_looking_east%2C_NYC.jpg/320px-Southwest_corner_of_Central_Park%2C_looking_east%2C_NYC.jpg",
    "label": "New York City skyline with Central Park",
},
```

**If two similar images get confused:** The fix is almost always in the caption, not the retrieval. Make sure the captions explicitly describe what is *different* between the two images.

### Exercise 3 — Improve Caption Quality for a Hard Query

**Sample before/after for a color query:**

**Hard query:** *"Which images have a yellow or golden subject?"*

**Before (generic caption):**
```
[dog] A dog is shown in an outdoor setting.
```
This caption scores poorly against "yellow or golden" — the text embedding has no color signal at all.

**After (rich caption with explicit color):**
```
[dog] A golden-yellow Labrador Retriever with a warm honey-colored coat sits on a
green lawn in bright sunlight. The dog's fur displays varying shades of gold, cream,
and amber. Its warm brown eyes and dark nose provide strong contrast against the
bright yellow coat. The background is a lush green garden setting.
```

Now a query for "yellow or golden" retrieves the dog first with high similarity, because "golden", "yellow", "gold", "amber", and "honey" all appear in the caption and embed close to the query.

**General principle:** Every visual attribute you want to be queryable must appear *explicitly* as text in the caption. The embedding model cannot infer attributes that were never written down.